# KDAL V22 1 PM Direct Bucket Challenger

Two-stage direct bucket correction: predict whether the V20 1 PM point bucket is wrong, then classify the actual bucket as lower, same, or upper. Overrides require stable positive lift in both forward years.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src/calibration/bucket_correction.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT: raise RuntimeError('Project root not found')
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON = PROJECT_ROOT / '.venv/Scripts/python.exe'
POINT_DIR = PROJECT_ROOT / 'data/calibration/station_stacking_v20_kdal_1pm_no_peak'
OUTPUT_DIR = PROJECT_ROOT / 'data/calibration/station_stacking_v22_kdal_1pm_direct_bucket'
POINT_VERSION = 'station_high_regressor_v20_kdal_1pm_no_peak_stack'
BUCKET_VERSION = 'station_bucket_v22_kdal_1pm_direct'
RUN_TRAINING = False
PROJECT_ROOT


## Verify immutable point contract


In [ ]:
point_manifest = json.loads((POINT_DIR / 'model_weights' / f'KDAL_{POINT_VERSION}.json').read_text(encoding='utf-8'))
contract = point_manifest['model_contract']
assert contract['timing_mode'] == 'same_day_1pm_live_safe'
assert contract['feature_version'] == 'v20_kdal_1pm_no_peak'
assert contract['target_mode'] == 'remaining_warmup'
contract


## Train direct lower/same/upper challenger


In [ ]:
manifest_path = OUTPUT_DIR / 'model_weights' / f'KDAL_{BUCKET_VERSION}.json'
if RUN_TRAINING or not manifest_path.exists():
    subprocess.run([
        str(PYTHON), str(PROJECT_ROOT / 'scripts/train-bucket-correction.py'),
        '--station', 'KDAL', '--pipeline-dir', str(POINT_DIR),
        '--point-bundle', str(POINT_DIR / 'model_weights' / f'KDAL_{POINT_VERSION}.joblib'),
        '--point-model-version', POINT_VERSION, '--model-version', BUCKET_VERSION,
        '--feature-profile', 'kdal_1pm', '--output-dir', str(OUTPUT_DIR),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Using existing V22 artifacts; set RUN_TRAINING=True to retrain.')


## Full mismatch audit


In [ ]:
subprocess.run([str(PYTHON), str(PROJECT_ROOT / 'scripts/audit_v22_kdal_1pm_direct_bucket.py')], cwd=PROJECT_ROOT, check=True)
audit = json.loads((OUTPUT_DIR / 'audit/audit_result.json').read_text(encoding='utf-8'))
assert audit['passed']
audit


## Forward and 2026 results


In [ ]:
forward = pd.read_csv(OUTPUT_DIR / 'KDAL_forward_bucket_correction_metrics.csv')
holdout = pd.read_csv(OUTPUT_DIR / 'KDAL_2026_bucket_correction_holdout_metrics.csv')
display(forward)
display(holdout)


## Promotion decision


In [ ]:
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
acceptance = manifest['historical_acceptance']
if acceptance['passed']:
    print('Historical gates passed; shadow evaluation is next.')
else:
    print('RESEARCH-ONLY:', ', '.join(acceptance['reasons']))
acceptance
